# Fort Collins Levy Policy Example

This notebook uses the same general narrative structure as the Baltimore and Cleveland examples, adapted to Larimer County's parcel data for Fort Collins, Colorado.


In [ ]:
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

sys.path.insert(0, str(Path('../..').resolve()))
REPO_ROOT = Path('../..').resolve()

from lvt_utils import calculate_category_tax_summary
from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from policy_analysis import analyze_vacant_land, analyze_land_by_improvement_share
from viz import (
    calculate_block_group_summary,
    create_quintile_summary,
    create_spokane_property_category_chart,
    create_threshold_change_chart,
    filter_data_for_analysis,
    plot_upside_down_quintile_bars,
)

load_dotenv(REPO_ROOT / ".env")
sns.set_theme(style="white")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

ACCOUNT_URL = "https://storage.googleapis.com/lc-public/asr/assessor-public-account.csv"
VALUE_URL = "https://storage.googleapis.com/lc-public/asr/assessor-public-value-detail.csv"
IMPROVEMENT_URL = "https://storage.googleapis.com/lc-public/asr/assessor-public-improvement.csv"
GEOMETRY_URL = "zip+https://apps.larimer.org/api/gis/files/GIS_ParcelOwnerSHP.zip"
BUILD_PROPINFO_CACHE = False


In [ ]:
data_dir = Path('data')
data_dir.mkdir(parents=True, exist_ok=True)
data_dir


## Step 1: Load Parcel Data

The account table gives parcel location, tax area, and levy fields. The value-detail table gives `Land` and `Improvement` rows with actual value plus separate assessed-value bases for local and school levies.


In [ ]:
parcel_attrs = pd.read_csv(
    ACCOUNT_URL,
    usecols=[
        "SCHEDULENUM",
        "PARCELNO",
        "TAXAREA",
        "ACCTTYPE",
        "APPRAISALTYPE",
        "SITUSADDRESS",
        "SITUSCITY",
        "SITUSZIPCODE",
        "LGMILLLEVY",
        "SCHOOLMILLLEVY",
        "TOTALMILLLEVY",
        "TAXYEAR",
        "RUNDATE",
    ],
)

fort_collins = parcel_attrs[
    (parcel_attrs["APPRAISALTYPE"] == "Real")
    & (parcel_attrs["SITUSCITY"] == "FORT COLLINS")
].copy()

print(f"Total Fort Collins parcels: {len(fort_collins):,}")
fort_collins.head()


In [ ]:
fc_sched = set(fort_collins["SCHEDULENUM"].astype(int).tolist())

manual_override_path = data_dir / "larimer_manual_school_assessed_value_overrides_2026-03-31.csv"
manual_school_overrides = pd.read_csv(
    manual_override_path,
    usecols=["accountno", "live_api_improvement_school_assessed_value"],
)
manual_school_overrides["SCHEDULENUM"] = pd.to_numeric(
    manual_school_overrides["accountno"].str.replace("R", "", regex=False),
    errors="coerce",
).astype("Int64")
manual_school_overrides["live_api_improvement_school_assessed_value"] = pd.to_numeric(
    manual_school_overrides["live_api_improvement_school_assessed_value"],
    errors="coerce",
)
manual_school_override_map = (
    manual_school_overrides.dropna(subset=["SCHEDULENUM", "live_api_improvement_school_assessed_value"])
    .drop_duplicates(subset=["SCHEDULENUM"])
    .set_index("SCHEDULENUM")["live_api_improvement_school_assessed_value"]
)

value_parts = []
for chunk in pd.read_csv(
    VALUE_URL,
    usecols=["SCHEDULENUM", "VALUETYPE", "ACTUALVALUE", "LG_ASDVALUE", "SCHOOL_ASDVALUE"],
    chunksize=250000,
):
    chunk = chunk[chunk["SCHEDULENUM"].isin(fc_sched)]
    if len(chunk):
        value_parts.append(
            chunk.groupby(["SCHEDULENUM", "VALUETYPE"], dropna=False)[
                ["ACTUALVALUE", "LG_ASDVALUE", "SCHOOL_ASDVALUE"]
            ]
            .sum()
            .reset_index()
        )

value_detail = (
    pd.concat(value_parts, ignore_index=True)
    .groupby(["SCHEDULENUM", "VALUETYPE"], dropna=False)[["ACTUALVALUE", "LG_ASDVALUE", "SCHOOL_ASDVALUE"]]
    .sum()
    .reset_index()
)

override_mask = (
    value_detail["VALUETYPE"].eq("Improvement")
    & value_detail["SCHEDULENUM"].isin(manual_school_override_map.index)
)
value_detail.loc[override_mask, "SCHOOL_ASDVALUE"] = (
    value_detail.loc[override_mask, "SCHEDULENUM"].map(manual_school_override_map)
)
print(f"Applied manual school assessed value overrides to {int(override_mask.sum())} Fort Collins improvement rows.")

value_pivot = value_detail.pivot(
    index="SCHEDULENUM",
    columns="VALUETYPE",
    values=["ACTUALVALUE", "LG_ASDVALUE", "SCHOOL_ASDVALUE"],
).fillna(0)
value_pivot.columns = [f"{a.lower()}_{b.lower()}" for a, b in value_pivot.columns]
value_pivot = value_pivot.reset_index().rename(
    columns={
        "actualvalue_land": "land_actual_value",
        "actualvalue_improvement": "improvement_actual_value",
        "lg_asdvalue_land": "land_lg_asd",
        "lg_asdvalue_improvement": "improvement_lg_asd",
        "school_asdvalue_land": "land_school_asd",
        "school_asdvalue_improvement": "improvement_school_asd",
    }
)

for col in [
    "land_actual_value",
    "improvement_actual_value",
    "land_lg_asd",
    "improvement_lg_asd",
    "land_school_asd",
    "improvement_school_asd",
]:
    if col not in value_pivot.columns:
        value_pivot[col] = 0.0

fort_collins = fort_collins.merge(value_pivot, on="SCHEDULENUM", how="left").fillna(0)

propinfo_cache_path = data_dir / "fort_collins_propinfo_cache.csv"
propinfo_builder_script = REPO_ROOT / "scripts" / "build_fort_collins_propinfo_cache.py"
tax_year = int(fort_collins["TAXYEAR"].mode().iat[0])

if BUILD_PROPINFO_CACHE or not propinfo_cache_path.exists():
    subprocess.run(
        [
            sys.executable,
            str(propinfo_builder_script),
            "--cache-path",
            str(propinfo_cache_path),
        ],
        check=True,
        cwd=REPO_ROOT,
    )

propinfo_df = pd.read_csv(propinfo_cache_path)
propinfo_df = propinfo_df[propinfo_df["tax_year"] == tax_year].copy()
fort_collins = fort_collins.merge(propinfo_df, on="SCHEDULENUM", how="left")
fort_collins[["OWNER_TAX_LIABILITY", "STATE_TAX_LIABILITY"]] = fort_collins[["OWNER_TAX_LIABILITY", "STATE_TAX_LIABILITY"]].fillna(0)
fort_collins["full_tax_bill"] = fort_collins["OWNER_TAX_LIABILITY"] + fort_collins["STATE_TAX_LIABILITY"]
fort_collins["owner_tax_share"] = np.where(
    fort_collins["full_tax_bill"] > 0,
    fort_collins["OWNER_TAX_LIABILITY"] / fort_collins["full_tax_bill"],
    1.0,
)
fort_collins["state_tax_relief_pct"] = 1.0 - fort_collins["owner_tax_share"]

fort_collins["effective_land_lg_asd"] = fort_collins["land_lg_asd"] * fort_collins["owner_tax_share"]
fort_collins["effective_improvement_lg_asd"] = fort_collins["improvement_lg_asd"] * fort_collins["owner_tax_share"]
fort_collins["effective_land_school_asd"] = fort_collins["land_school_asd"] * fort_collins["owner_tax_share"]
fort_collins["effective_improvement_school_asd"] = fort_collins["improvement_school_asd"] * fort_collins["owner_tax_share"]

excluded_fully_exempt = int((fort_collins["ACCTTYPE"] == "Exempt").sum())
fort_collins = fort_collins[fort_collins["ACCTTYPE"] != "Exempt"].copy()

fort_collins["total_actual_value"] = fort_collins["land_actual_value"] + fort_collins["improvement_actual_value"]
fort_collins["land_share_actual"] = np.where(
    fort_collins["total_actual_value"] > 0,
    fort_collins["land_actual_value"] / fort_collins["total_actual_value"],
    np.nan,
)

print(
    f"Loaded treasurer relief data for {(fort_collins['full_tax_bill'] > 0).sum():,} Fort Collins parcels; "
    f"mean state relief share = {fort_collins['state_tax_relief_pct'].mean():.4%}. "
    f"Excluded {excluded_fully_exempt:,} fully exempt parcels from analysis."
)
fort_collins.head()


## Step 2: Prepare Fort Collins Parcels for Modeling

The existing examples use a common property-category field for summaries and charts. Here we create a Fort Collins version from `ACCTTYPE`, the parcel's land/improvement pattern, and the primary improvement record from Larimer's public improvement export.


In [ ]:
improvement_parts = []
for chunk in pd.read_csv(
    IMPROVEMENT_URL,
    usecols=[
        "SCHEDULENUM",
        "IMPACTUALVALUE",
        "IMPNO",
        "PROPERTYTYPE",
        "OCCDESCRIPTION",
        "BLTASDESCRIPTION",
    ],
    chunksize=250000,
    low_memory=False,
):
    chunk = chunk[chunk["SCHEDULENUM"].isin(fc_sched)].copy()
    if len(chunk):
        improvement_parts.append(chunk)

improvement_df = pd.concat(improvement_parts, ignore_index=True)
improvement_df["IMPACTUALVALUE"] = pd.to_numeric(improvement_df["IMPACTUALVALUE"], errors="coerce").fillna(0)
improvement_df["IMPNO"] = pd.to_numeric(improvement_df["IMPNO"], errors="coerce").fillna(0)
improvement_df["PROPERTYTYPE"] = improvement_df["PROPERTYTYPE"].fillna("")
improvement_df["OCCDESCRIPTION"] = improvement_df["OCCDESCRIPTION"].fillna("")
improvement_df["BLTASDESCRIPTION"] = improvement_df["BLTASDESCRIPTION"].fillna("")

primary_improvement = (
    improvement_df.sort_values(["SCHEDULENUM", "IMPACTUALVALUE", "IMPNO"], ascending=[True, False, True])
    .drop_duplicates(subset=["SCHEDULENUM"], keep="first")
    [["SCHEDULENUM", "PROPERTYTYPE", "OCCDESCRIPTION", "BLTASDESCRIPTION"]]
    .rename(
        columns={
            "PROPERTYTYPE": "primary_property_type",
            "OCCDESCRIPTION": "primary_occ_description",
            "BLTASDESCRIPTION": "primary_bltas_description",
        }
    )
)

surface_parking_flags = (
    improvement_df.assign(
        is_surface_parking_lot=(
            improvement_df["OCCDESCRIPTION"].str.contains(r"\bParking Lot\b", case=False, na=False)
            | improvement_df["BLTASDESCRIPTION"].str.contains(r"\bParking Lot\b", case=False, na=False)
        )
    )
    .groupby("SCHEDULENUM", as_index=False)["is_surface_parking_lot"]
    .max()
)

fort_collins = fort_collins.merge(primary_improvement, on="SCHEDULENUM", how="left")
fort_collins = fort_collins.merge(surface_parking_flags, on="SCHEDULENUM", how="left")
fort_collins["is_surface_parking_lot"] = fort_collins["is_surface_parking_lot"].fillna(False)

def categorize_fort_collins_property(row):
    if row["improvement_actual_value"] == 0 and row["land_actual_value"] > 0:
        return "Vacant Land"

    acct = row["ACCTTYPE"]
    property_type = str(row.get("primary_property_type", "") or "")
    occ = str(row.get("primary_occ_description", "") or "")

    if row.get("is_surface_parking_lot", False) and acct in {"Commercial", "PI Comm"}:
        return "Surface Parking Lot"

    if acct in {"Residential", "Partial Exempt"}:
        if property_type == "Condo" or "Condo unit" in occ:
            return "Condo"
        if property_type in {"Townhouse", "Duplex", "Triplex"} or "Townhouse" in occ or "Duplex" in occ or "Triplex" in occ:
            return "Townhouse / Duplex"
        if "Apartment w/4-8 Units" in occ:
            return "4-8 Unit Multifamily"
        if "Apartment w/9 + Units" in occ:
            return "9+ Unit Multifamily"
        return "Single Family Residential"

    if acct == "Multiple Unit":
        if property_type in {"Duplex", "Triplex", "Townhouse"} or "Duplex" in occ or "Triplex" in occ or "Townhouse" in occ:
            return "Townhouse / Duplex"
        if "Apartment w/4-8 Units" in occ:
            return "4-8 Unit Multifamily"
        return "9+ Unit Multifamily"

    if acct == "Mobile Home":
        return "Mobile Home"
    if acct == "MH Park":
        return "Mobile Home Park"
    if acct in {"Commercial", "PI Comm"}:
        return "Commercial / Mixed Use"
    if acct == "Industrial":
        return "Industrial"
    if acct == "Agricultural":
        return "Agricultural"
    if acct == "Nat Resources":
        return "Natural Resources"
    return "Other"

fort_collins["PROPERTY_CATEGORY"] = fort_collins.apply(categorize_fort_collins_property, axis=1)

fort_collins["PROPERTY_CATEGORY"].value_counts().rename_axis("PROPERTY_CATEGORY").to_frame("count")


## Step 3: Recreate Current Fort Collins Taxes

Current tax is calculated from the county's separate assessed bases for local and school levies:

```text
current_tax_local  = (land_lg_asd + improvement_lg_asd) * LGMILLLEVY / 1000
current_tax_school = (land_school_asd + improvement_school_asd) * SCHOOLMILLLEVY / 1000
current_tax_total  = current_tax_local + current_tax_school
```


In [ ]:
fort_collins["current_tax_local_full"] = (
    (fort_collins["land_lg_asd"] + fort_collins["improvement_lg_asd"]) *
    fort_collins["LGMILLLEVY"] / 1000.0
)
fort_collins["current_tax_school_full"] = (
    (fort_collins["land_school_asd"] + fort_collins["improvement_school_asd"]) *
    fort_collins["SCHOOLMILLLEVY"] / 1000.0
)
fort_collins["current_tax_local"] = (
    (fort_collins["effective_land_lg_asd"] + fort_collins["effective_improvement_lg_asd"]) *
    fort_collins["LGMILLLEVY"] / 1000.0
)
fort_collins["current_tax_school"] = (
    (fort_collins["effective_land_school_asd"] + fort_collins["effective_improvement_school_asd"]) *
    fort_collins["SCHOOLMILLLEVY"] / 1000.0
)
fort_collins["current_tax"] = fort_collins["current_tax_local"] + fort_collins["current_tax_school"]

pd.DataFrame(
    {
        "Metric": [
            "Real-property parcels modeled",
            "Tax year",
            "Tax areas represented",
            "Total actual value",
            "Total local assessed value",
            "Total school assessed value",
            "Current modeled tax revenue",
        ],
        "Value": [
            f"{len(fort_collins):,}",
            int(fort_collins["TAXYEAR"].mode().iat[0]),
            int(fort_collins["TAXAREA"].nunique()),
            fort_collins["total_actual_value"].sum(),
            fort_collins["land_lg_asd"].sum() + fort_collins["improvement_lg_asd"].sum(),
            fort_collins["land_school_asd"].sum() + fort_collins["improvement_school_asd"].sum(),
            fort_collins["current_tax"].sum(),
        ],
    }
)


## Step 4: Modeling the Split-Rate Land Value Tax

To preserve Fort Collins' levy geography, each scenario is solved separately inside each `TAXAREA` for local and school components, then recombined parcel by parcel.


In [ ]:
def run_split_rate_scenario(df_input, ratio):
    df = df_input.copy()

    local_rates = df.groupby("TAXAREA").agg(
        revenue=("current_tax_local", "sum"),
        land=("effective_land_lg_asd", "sum"),
        imp=("effective_improvement_lg_asd", "sum"),
        current_mill=("LGMILLLEVY", "first"),
    )
    local_rates["improvement_mill_new"] = np.where(
        (ratio * local_rates["land"] + local_rates["imp"]) > 0,
        local_rates["revenue"] * 1000 / (ratio * local_rates["land"] + local_rates["imp"]),
        0,
    )
    local_rates["land_mill_new"] = ratio * local_rates["improvement_mill_new"]
    local_rates = local_rates.reset_index()

    school_rates = df.groupby("TAXAREA").agg(
        revenue=("current_tax_school", "sum"),
        land=("effective_land_school_asd", "sum"),
        imp=("effective_improvement_school_asd", "sum"),
        current_mill=("SCHOOLMILLLEVY", "first"),
    )
    school_rates["improvement_mill_new"] = np.where(
        (ratio * school_rates["land"] + school_rates["imp"]) > 0,
        school_rates["revenue"] * 1000 / (ratio * school_rates["land"] + school_rates["imp"]),
        0,
    )
    school_rates["land_mill_new"] = ratio * school_rates["improvement_mill_new"]
    school_rates = school_rates.reset_index()

    df = df.merge(
        local_rates[["TAXAREA", "land_mill_new", "improvement_mill_new"]],
        on="TAXAREA",
        how="left",
    ).rename(
        columns={
            "land_mill_new": "local_land_mill_new",
            "improvement_mill_new": "local_imp_mill_new",
        }
    )
    df = df.merge(
        school_rates[["TAXAREA", "land_mill_new", "improvement_mill_new"]],
        on="TAXAREA",
        how="left",
    ).rename(
        columns={
            "land_mill_new": "school_land_mill_new",
            "improvement_mill_new": "school_imp_mill_new",
        }
    )

    df["new_tax_local_full"] = (
        df["land_lg_asd"] * df["local_land_mill_new"] / 1000 +
        df["improvement_lg_asd"] * df["local_imp_mill_new"] / 1000
    )
    df["new_tax_school_full"] = (
        df["land_school_asd"] * df["school_land_mill_new"] / 1000 +
        df["improvement_school_asd"] * df["school_imp_mill_new"] / 1000
    )
    df["new_tax_local"] = (
        df["effective_land_lg_asd"] * df["local_land_mill_new"] / 1000 +
        df["effective_improvement_lg_asd"] * df["local_imp_mill_new"] / 1000
    )
    df["new_tax_school"] = (
        df["effective_land_school_asd"] * df["school_land_mill_new"] / 1000 +
        df["effective_improvement_school_asd"] * df["school_imp_mill_new"] / 1000
    )
    df["new_tax"] = df["new_tax_local"] + df["new_tax_school"]
    df["tax_change"] = df["new_tax"] - df["current_tax"]
    df["tax_change_pct"] = np.where(
        df["current_tax"] > 0,
        df["tax_change"] / df["current_tax"] * 100,
        0,
    )

    category_summary = calculate_category_tax_summary(
        df,
        category_col="PROPERTY_CATEGORY",
        current_tax_col="current_tax",
        new_tax_col="new_tax",
        pct_threshold=10.0,
    )

    return {
        "ratio": ratio,
        "scenario": f"Split-rate {int(ratio)}:1",
        "df": df,
        "local_rates": local_rates,
        "school_rates": school_rates,
        "category_summary": category_summary,
    }


scenarios = [run_split_rate_scenario(fort_collins, 2.0), run_split_rate_scenario(fort_collins, 4.0)]

pd.DataFrame(
    {
        "scenario": [s["scenario"] for s in scenarios],
        "current_revenue": [s["df"]["current_tax"].sum() for s in scenarios],
        "new_revenue": [s["df"]["new_tax"].sum() for s in scenarios],
        "revenue_difference": [s["df"]["new_tax"].sum() - s["df"]["current_tax"].sum() for s in scenarios],
        "median_tax_change": [s["df"]["tax_change"].median() for s in scenarios],
        "pct_parcels_up": [(s["df"]["tax_change"] > 1e-9).mean() * 100 for s in scenarios],
        "pct_parcels_down": [(s["df"]["tax_change"] < -1e-9).mean() * 100 for s in scenarios],
    }
)


In [ ]:
fort_collins_2to1 = next(s for s in scenarios if s["scenario"] == "Split-rate 2:1")["df"].copy()
category_summary = next(s for s in scenarios if s["scenario"] == "Split-rate 2:1")["category_summary"].copy()
category_summary


## Step 5: Property Category Impact Visualizations


In [ ]:
create_spokane_property_category_chart(
    category_summary,
    title="Fort Collins 2:1 Split-Rate Tax Impact by Property Category",
    min_count=25,
)


In [ ]:
create_threshold_change_chart(
    category_summary,
    title="Fort Collins 2:1 Split-Rate: Share of Parcels with Tax Changes Greater than 10%",
    threshold=10.0,
    min_count=25,
)


In [ ]:
focus_categories = [
    "Single Family Residential",
    "Condo",
    "Townhouse / Duplex",
    "4-8 Unit Multifamily",
    "9+ Unit Multifamily",
    "Commercial / Mixed Use",
    "Surface Parking Lot",
    "Industrial",
    "Vacant Land",
]

scenario_rows = []
for scenario in scenarios:
    summary = scenario["category_summary"].copy()
    summary = summary[summary["PROPERTY_CATEGORY"].isin(focus_categories)]
    summary["scenario"] = scenario["scenario"]
    scenario_rows.append(summary[["scenario", "PROPERTY_CATEGORY", "median_tax_change_pct"]])

scenario_category_df = pd.concat(scenario_rows, ignore_index=True)
pivot = scenario_category_df.pivot(
    index="scenario",
    columns="PROPERTY_CATEGORY",
    values="median_tax_change_pct",
)
display(pivot)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind="bar", ax=ax, color=["#2E8B57", "#66A61E", "#E67E22", "#B22222", "#8B0000"])
ax.axhline(0, color="black", linestyle="dotted", linewidth=1)
ax.set_ylabel("Median tax change (%)")
ax.set_title("Scenario Comparison for Key Fort Collins Property Categories")
ax.legend(title="Property Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
sample = fort_collins_2to1[
    (fort_collins_2to1["total_actual_value"] > 0) &
    np.isfinite(fort_collins_2to1["tax_change_pct"])
].copy()
if len(sample) > 8000:
    sample = sample.sample(8000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    sample["land_share_actual"],
    sample["tax_change_pct"].clip(-100, 200),
    s=10,
    alpha=0.25,
    color="#1f77b4",
    edgecolors="none",
)
ax.axhline(0, color="black", linestyle="dotted", linewidth=1)
ax.set_xlabel("Land share of actual parcel value")
ax.set_ylabel("Tax change (%) clipped to [-100, 200]")
ax.set_title("Fort Collins 2:1 Split-Rate: Land Share vs. Tax Change")
plt.tight_layout()
plt.show()


## Step 6: Vacant Land and Underused-Land Diagnostics


In [ ]:
vacant_results = analyze_vacant_land(
    fort_collins_2to1,
    land_value_col="land_actual_value",
    property_type_col="PROPERTY_CATEGORY",
    vacant_identifier="Vacant Land",
    improvement_value_col="improvement_actual_value",
)

print("Vacant parcels:", vacant_results.get("total_vacant_parcels"))
print("Total vacant land value:", vacant_results.get("total_vacant_land_value"))
print("Vacant land share of all city land value (%):", vacant_results.get("vacant_land_pct_of_total"))

top_vacant = fort_collins_2to1[fort_collins_2to1["PROPERTY_CATEGORY"] == "Vacant Land"].copy()
top_vacant = top_vacant.sort_values("land_actual_value", ascending=False)[
    ["SCHEDULENUM", "SITUSADDRESS", "TAXAREA", "land_actual_value", "current_tax", "new_tax", "tax_change"]
].head(10)
top_vacant


In [ ]:
vacant_by_tax_area = (
    fort_collins_2to1[fort_collins_2to1["PROPERTY_CATEGORY"] == "Vacant Land"]
    .groupby("TAXAREA")
    .agg(
        parcel_count=("SCHEDULENUM", "count"),
        land_value=("land_actual_value", "sum"),
        mean_tax_change=("tax_change", "mean"),
    )
    .sort_values("land_value", ascending=False)
    .head(10)
    .reset_index()
)
display(vacant_by_tax_area)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(vacant_by_tax_area["TAXAREA"].astype(str), vacant_by_tax_area["land_value"], color="#8B4513")
ax.set_title("Top Fort Collins Tax Areas by Vacant Land Value")
ax.set_xlabel("Tax Area")
ax.set_ylabel("Vacant land actual value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
underused = analyze_land_by_improvement_share(
    fort_collins_2to1,
    land_value_col="land_actual_value",
    improvement_value_col="improvement_actual_value",
)

underused_df = pd.DataFrame(underused["categories"])
underused_df


In [ ]:
slide_summary = {
    "vacant_land_value": vacant_results["total_vacant_land_value"],
    "vacant_land_pct": vacant_results["vacant_land_pct_of_total"],
    "extremely_underdeveloped_value": underused_df.loc[
        underused_df["category"] == "<10% improvement (excl. 0%)", "adjusted_land_value"
    ].iloc[0],
    "extremely_underdeveloped_pct": underused_df.loc[
        underused_df["category"] == "<10% improvement (excl. 0%)", "share_of_total_land_value_pct"
    ].iloc[0],
    "highly_underdeveloped_value": underused_df.loc[
        underused_df["category"] == "10-25% improvement", "adjusted_land_value"
    ].iloc[0],
    "highly_underdeveloped_pct": underused_df.loc[
        underused_df["category"] == "10-25% improvement", "share_of_total_land_value_pct"
    ].iloc[0],
    "underdeveloped_value": underused_df.loc[
        underused_df["category"] == "25-50% improvement", "adjusted_land_value"
    ].iloc[0],
    "underdeveloped_pct": underused_df.loc[
        underused_df["category"] == "25-50% improvement", "share_of_total_land_value_pct"
    ].iloc[0],
}
slide_summary["underutilized_value"] = (
    slide_summary["extremely_underdeveloped_value"]
    + slide_summary["highly_underdeveloped_value"]
    + slide_summary["underdeveloped_value"]
)
slide_summary["underutilized_pct"] = (
    slide_summary["extremely_underdeveloped_pct"]
    + slide_summary["highly_underdeveloped_pct"]
    + slide_summary["underdeveloped_pct"]
)
slide_summary["combined_value"] = (
    slide_summary["vacant_land_value"] + slide_summary["underutilized_value"]
)
slide_summary["combined_pct"] = (
    slide_summary["vacant_land_pct"] + slide_summary["underutilized_pct"]
)

for label, value in [
    ("Vacant land value", slide_summary["vacant_land_value"]),
    ("Vacant land share", slide_summary["vacant_land_pct"]),
    ("Extremely underdeveloped value", slide_summary["extremely_underdeveloped_value"]),
    ("Extremely underdeveloped share", slide_summary["extremely_underdeveloped_pct"]),
    ("Highly underdeveloped value", slide_summary["highly_underdeveloped_value"]),
    ("Highly underdeveloped share", slide_summary["highly_underdeveloped_pct"]),
    ("Underdeveloped value", slide_summary["underdeveloped_value"]),
    ("Underdeveloped share", slide_summary["underdeveloped_pct"]),
    ("Underutilized value", slide_summary["underutilized_value"]),
    ("Underutilized share", slide_summary["underutilized_pct"]),
    ("Combined value", slide_summary["combined_value"]),
    ("Combined share", slide_summary["combined_pct"]),
]:
    suffix = "%" if "share" in label.lower() else ""
    print(f"{label}: ${value:,.0f}" if not suffix else f"{label}: {value:.1f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    underused_df["category"],
    underused_df["share_of_total_land_value_pct"],
    color="#6A5ACD",
)
ax.set_title("Share of Fort Collins Land Value in Low-Improvement Parcels")
ax.set_ylabel("Share of total land value (%)")
ax.set_xlabel("Improvement-share category")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## Step 7: Census Progressivity Analysis

This follows the stronger Cleveland / Spokane pattern, using parcel-level tax changes joined to Census block groups and then summarized into quintiles.


In [ ]:
parcel_geo = gpd.read_file(GEOMETRY_URL)
parcel_geo["SCHEDNUM"] = pd.to_numeric(parcel_geo["SCHEDNUM"], errors="coerce")
if parcel_geo.crs is None:
    parcel_geo = parcel_geo.set_crs("EPSG:2876")
fort_collins_geo = parcel_geo[parcel_geo["SCHEDNUM"].isin(fc_sched)].copy()

fort_collins_geo = gpd.GeoDataFrame(
    fort_collins_geo.merge(
        fort_collins_2to1,
        left_on="SCHEDNUM",
        right_on="SCHEDULENUM",
        how="inner",
    ),
    geometry="geometry",
    crs=parcel_geo.crs,
)
fort_collins_geo = fort_collins_geo.to_crs(3857)

print(f"Geometry rows merged: {len(fort_collins_geo):,}")
fort_collins_geo[["SCHEDNUM", "LOCADDRESS", "PROPERTY_CATEGORY", "tax_change_pct"]].head()


In [ ]:
census_data, census_boundaries = get_census_data_with_boundaries("08069", year=2022)
if census_boundaries.crs is None:
    census_boundaries = census_boundaries.set_crs("EPSG:4326")
df_geo = match_to_census_blockgroups(fort_collins_geo, census_boundaries)

print(f"Parcels matched to Census block groups: {len(df_geo):,}")
df_geo[["std_geoid", "median_income", "minority_pct", "black_pct", "PROPERTY_CATEGORY", "tax_change_pct"]].head()


In [ ]:
gdf_filtered, non_vacant_gdf = filter_data_for_analysis(df_geo)

non_vacant_income_quintile_summary = create_quintile_summary(
    non_vacant_gdf,
    "median_income",
    "median_income",
)
non_vacant_minority_quintile_summary = create_quintile_summary(
    non_vacant_gdf,
    "minority_pct",
    "minority_pct",
)

residential_categories = [
    "Single Family Residential",
]
non_vacant_residential_gdf = non_vacant_gdf[
    non_vacant_gdf["PROPERTY_CATEGORY"].isin(residential_categories)
].copy()

non_vacant_income_quintile_summary_res = create_quintile_summary(
    non_vacant_residential_gdf,
    "median_income",
    "median_income",
)
non_vacant_minority_quintile_summary_res = create_quintile_summary(
    non_vacant_residential_gdf,
    "minority_pct",
    "minority_pct",
)

print("Non-vacant income quintiles")
display(non_vacant_income_quintile_summary)
print("Non-vacant minority-share quintiles")
display(non_vacant_minority_quintile_summary)
print("Residential-only non-vacant income quintiles")
display(non_vacant_income_quintile_summary_res)
print("Residential-only non-vacant minority-share quintiles")
display(non_vacant_minority_quintile_summary_res)


In [ ]:
plot_upside_down_quintile_bars(
    non_vacant_income_quintile_summary,
    "Median Tax Change by Neighborhood Median Income (Excl. Vacant Land)",
)


In [ ]:
plot_upside_down_quintile_bars(
    non_vacant_minority_quintile_summary,
    "Median Tax Change by Minority Percentage Quintile (Excl. Vacant Land)",
)


In [ ]:
plot_upside_down_quintile_bars(
    non_vacant_income_quintile_summary_res,
    "Median Tax Change by Neighborhood Median Income (Excl. Vacant Land, Residential Only)",
)


In [ ]:
plot_upside_down_quintile_bars(
    non_vacant_minority_quintile_summary_res,
    "Median Tax Change by Minority Percentage Quintile (Excl. Vacant Land, Residential Only)",
)


In [ ]:
bg_summary = calculate_block_group_summary(df_geo)
bg_map = census_boundaries.merge(
    bg_summary[["std_geoid", "mean_tax_change_pct", "median_income", "minority_pct", "parcel_count"]],
    on="std_geoid",
    how="left",
)

fig, ax = plt.subplots(figsize=(10, 10))
bg_map.plot(
    column="mean_tax_change_pct",
    cmap="RdYlGn_r",
    linewidth=0.15,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "#d9d9d9", "label": "No data"},
)
ax.set_title("Mean Parcel Tax Change by Census Block Group (Fort Collins 2:1)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


## Step 8: Adding Geographic Context

Fort Collins parcel geometry comes from Larimer County's public `GIS_ParcelOwnerSHP.zip` download.


In [ ]:
fort_collins_geo[["SCHEDNUM", "LOCADDRESS", "PROPERTY_CATEGORY", "tax_change_pct"]].head()


In [ ]:
map_df = fort_collins_geo.copy()
lo = map_df["tax_change_pct"].quantile(0.05)
hi = map_df["tax_change_pct"].quantile(0.95)
map_df["map_tax_change_pct"] = map_df["tax_change_pct"].clip(lo, hi)

fig, ax = plt.subplots(figsize=(10, 10))
map_df.plot(
    column="map_tax_change_pct",
    cmap="RdYlGn_r",
    linewidth=0,
    legend=True,
    ax=ax,
)
ax.set_title("Fort Collins 2:1 Split-Rate: Parcel Tax Change Map")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
taxarea_geo = fort_collins_geo.dissolve(
    by="TAXAREA",
    aggfunc={
        "tax_change_pct": "mean",
        "current_tax": "sum",
        "new_tax": "sum",
    },
).reset_index()

fig, ax = plt.subplots(figsize=(10, 10))
taxarea_geo.plot(
    column="tax_change_pct",
    cmap="RdYlGn_r",
    linewidth=0.25,
    edgecolor="white",
    legend=True,
    ax=ax,
)
ax.set_title("Mean Parcel Tax Change by Tax Area (Fort Collins 2:1)")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# Export standardized CSV — do not remove or move above Census join
import sys
sys.path.insert(0, '../..')
from lvt_utils import save_standard_export

# Compute combined effective taxable values (local + school portions)
df_geo['taxable_land_value'] = (
    df_geo.get('effective_land_lg_asd', 0) +
    df_geo.get('effective_land_school_asd', 0)
)
df_geo['taxable_improvement_value'] = (
    df_geo.get('effective_improvement_lg_asd', 0) +
    df_geo.get('effective_improvement_school_asd', 0)
)

# Revenue-weighted effective millage (median across parcels)
_land_mill = (df_geo.get('local_land_mill_new', 0) + df_geo.get('school_land_mill_new', 0)).median()
_imp_mill  = (df_geo.get('local_imp_mill_new', 0)  + df_geo.get('school_imp_mill_new', 0)).median()

out_df = save_standard_export(
    df=df_geo,
    city='fort_collins',
    output_path='../../analysis/data/fort_collins.csv',
    model_type='split_rate:2.0',
    land_millage=float(_land_mill),
    improvement_millage=float(_imp_mill),
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)


In [ ]:
# Standard report: category impact, income quintile, distribution
import sys
sys.path.insert(0, '../..')
from viz import create_city_report

create_city_report(out_df, 'fort_collins', show=True)
